# God of War Ragnarök — Ottimizzatore Build

Scraping dati armature e armi dalla wiki IGN, analisi inventario e piano di upgrade ottimo con vincoli risorse (Hacksilver + materiali).

**Come usare:** modifica la cella di configurazione (inventario + risorse), poi esegui tutto il notebook (Run All).

## Configurazione

Modifica qui il tuo inventario (armature e armi possedute con livello attuale) e le risorse disponibili.

In [31]:
# ============================================================
# CONFIGURAZIONE — Modifica qui il tuo inventario e risorse
# ============================================================

# --- Scraping vs CSV ---
# True = riscarica tutto dal web e aggiorna i CSV
# False = carica i CSV salvati (molto più veloce)
FORCE_SCRAPE = False

# --- Armature possedute (nome esatto, livello attuale) ---
# Formato: (nome, livello)         → pezzo già posseduto
#          (nome, livello, "craft") → pezzo da craftare (costo iniziale incluso)
chest_pieces = [
    ("Lunda's Lost Cuirass", 5, "craft"),
    ("Spiritual Shoulder Straps", 4, "craft"),
    ("Sol's Spaulders of Courage", 4),
    ("Spaulders of Enlightenment", 3, "craft"),
    ("Fortified Husk Cuirass", 3, "craft"),
    ("Nidavellir's Finest Plackart", 2, "craft"),
    ("Cloak of the Black Bear", 2),
    ("Vidar's Pauldron of Might", 1, "craft"),
    ("Shoulder Guard of Survival", 1),
    ("Spiritual Shoulder Straps", 4),
]

wrist_pieces = [
    ("Sol's Wraps of Courage", 4, "craft"),
    ("Gauntlets of Radiance", 4, "craft"),
    ("Spiritual Wraps", 5),
    ("Bracers of Enlightenment", 3, "craft"),
    ("Wraps of the Black Bear", 3),
    ("Nidavellir's Finest Arm Guards", 2, "craft"),
    ("Vidar's Bracers of Might", 1, "craft"),
    ("Fortified Husk Arm Guards", 1, "craft"),
    ("Wraps of Survival", 1),
]

waist_pieces = [
    ("Lunda's Lost Belt", 5, "craft"),
    ("Sol's Belt of Courage", 4, "craft"),
    ("Belt of Radiance", 4),
    ("Spiritual Belt", 5),
    ("Belt of Enlightenment", 3, "craft"),
    ("Belt of the Black Bear", 3),
    ("Nidavellir's Finest Waist Guard", 2, "craft"),
    ("Vidar's Belt of Might", 1, "craft"),
    ("Fortified Husk Girdle", 1, "craft"),
    ("Belt of Survival", 1),
]

# --- Armi possedute (nome esatto, livello attuale) ---
axe_attachments = [
    ("Grip of the Fallen Alchemist", 5),
    ("Grip of Weighted Recovery", 4, "craft"),
    ("Stonecutter's Knob", 4),
    ("Haur's Lucky Knob", 3),
    ("Wooden Knob", 3),
    ("The Furious Maul", 2, "craft"),
]

blades_attachments = [
    ("Hardened War Handles", 5),
    ("Pommels of Brutal Might", 4, "craft"),
    ("Pommels of Agile Deceit", 4),
    ("Steel Handles", 4),
    ("Radiant Warden Handles", 3),
]

spear_attachments = [
    # Draupnir Spear non ancora sbloccata
]

shield_attachments = [
    ("Rond of Affliction", 5),
    ("Rond of Expedition", 3),
    ("Rond of Volition", 1),
]

# --- Risorse disponibili (tutti i materiali del dataset) ---
resource_budget = {
    "Hacksilver":             2183,
    # ── Materiali base ──
    "Forged Iron":            60,
    "Dwarven Steel":          12,
    "Asgardian Ingot":        1,
    "Rawhide":                36,
    "Stonewood":              12,
    "Slag Deposits":          25,
    "Honed Metal":            10,
    # ── Materiali rari / boss ──
    "Bonded Leather":         35,
    "Dragon Tooth":           4,
    "Hardened Remnants":      0,
    "Tempered Remnants":      0,
    "Fortified Remnants":     0,
    "Petrified Bone":         0,
    "Lindwyrm Scales":        0,
    "Dragon Claw":            0,
    # ── Cristalli / Embers ──
    "Gleaming Crystal":       0,
    "Shining Crystal":        0,
    "Sparkling Crystal":      0,
    "Celestial Fossil":       0,
    "Glowing Embers":         0,
    "Smoldering Embers":      0,
    "Blazing Embers":         0,
    "Sovereign Coals":        0,
    # ── Materiali speciali ──
    "Whispering Slab":        52,
    "Nidavellir Ore":         2,
    "Dust of Realms":         0,
    "Essence of Hel":         0,
    "Luminous Alloy":         0,
    "Purified Crystalline":   0,
    "Forsaken Breath":        0,
    "Divine Ashes":           0,
    "Skap Slag":              0,
    "Mountain Root":          0,
    # ── Materiali quest Lunda ──
    "Lunda's Broken Cuirass": 0,
    "Lunda's Broken Bracers": 0,
    "Lunda's Broken Belt":    0,
}

print("Configurazione caricata.")
print(f"  Armature: {len(chest_pieces)} chest, {len(wrist_pieces)} wrist, {len(waist_pieces)} waist")
print(f"  Armi: {len(axe_attachments)} axe, {len(blades_attachments)} blades, "
      f"{len(spear_attachments)} spear, {len(shield_attachments)} shield")
craft_a = sum(1 for t in chest_pieces+wrist_pieces+waist_pieces if len(t)>2 and t[2]=="craft")
craft_w = sum(1 for t in axe_attachments+blades_attachments+spear_attachments+shield_attachments if len(t)>2 and t[2]=="craft")
print(f"  Da craftare: {craft_a} armature, {craft_w} armi")
print(f"  Hacksilver: {resource_budget['Hacksilver']:,}")
print(f"  FORCE_SCRAPE: {FORCE_SCRAPE}")

Configurazione caricata.
  Armature: 10 chest, 9 wrist, 10 waist
  Armi: 6 axe, 5 blades, 0 spear, 3 shield
  Da craftare: 18 armature, 3 armi
  Hacksilver: 2,183
  FORCE_SCRAPE: False


## Dati
Carica da CSV se disponibili, altrimenti scrapa il sito IGN e salva i CSV.

In [32]:
import os, time, re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

ARMOR_CSV = "all_pieces.csv"
WEAPONS_CSV = "all_weapons.csv"

stat_cols = ["Strength", "Defense", "Runic", "Vitality", "Cooldown", "Luck"]

if not FORCE_SCRAPE and os.path.exists(ARMOR_CSV) and os.path.exists(WEAPONS_CSV):
    # ─── Caricamento rapido da CSV ───
    all_pieces_df = pd.read_csv(ARMOR_CSV)
    all_weapons_df = pd.read_csv(WEAPONS_CSV)
    print(f"Caricati da CSV:")
    print(f"  Armature: {all_pieces_df.shape[0]} righe, {all_pieces_df['Piece Name'].nunique()} pezzi unici")
    print(f"  Armi:     {all_weapons_df.shape[0]} righe, {all_weapons_df['Weapon Name'].nunique()} armi uniche")

else:
    # ─── Scraping completo dal sito IGN ───
    print("=== SCRAPING DAL WEB ===\n")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    base_url = "https://www.ign.com"
    PIECE_NAMES = ["Chest", "Wrist", "Waist"]

    # --- Funzioni di parsing ---

    def parse_stats_text(cell):
        text = cell.get_text(separator="\n")
        pairs = re.findall(r'([A-Za-z][A-Za-z ]*?)\s*:\s*([\d,]+)', text)
        return {k.strip(): int(v.replace(",", "")) for k, v in pairs}

    def parse_detail_table(table_tag):
        rows = table_tag.find_all("tr")
        records = []
        current_levels = []
        i = 0
        while i < len(rows):
            cells = rows[i].find_all(["th", "td"])
            cell_texts = [c.get_text(strip=True) for c in cells]
            level_positions = [
                (col_idx, t)
                for col_idx, t in enumerate(cell_texts)
                if re.match(r'^Level\s+[\d.]+$', t)
            ]
            if level_positions:
                current_levels = level_positions
                i += 1
                continue
            if not current_levels or all(t == "" for t in cell_texts):
                i += 1
                continue
            full_text = " ".join(cell_texts)
            if "Strength:" in full_text or "Defense:" in full_text:
                for col_idx, level_name in current_levels:
                    if col_idx < len(cells):
                        stats = parse_stats_text(cells[col_idx])
                        if stats:
                            record = {"Level": level_name.replace("Level ", "")}
                            record.update(stats)
                            records.append(record)
                i += 1
                continue
            if "Hacksilver:" in full_text or "Upgrade" in full_text:
                for col_idx, level_name in current_levels:
                    if col_idx < len(cells):
                        costs = parse_stats_text(cells[col_idx])
                        level_val = level_name.replace("Level ", "")
                        for rec in records:
                            if rec["Level"] == level_val and "Upgrade_Hacksilver" not in rec:
                                for k, v in costs.items():
                                    rec[f"Upgrade_{k}"] = v
                                break
                i += 1
                continue
            i += 1
        return records

    def scrape_armor_detail(armor_url):
        resp = requests.get(armor_url, headers=headers)
        resp.raise_for_status()
        s = BeautifulSoup(resp.text, "lxml")
        tables = s.find_all("table")
        pieces = {}
        for idx, tbl in enumerate(tables):
            if idx >= 3:
                break
            piece_name = PIECE_NAMES[idx]
            prev_header = tbl.find_previous(["h2", "h3", "h4"])
            section = prev_header.get_text(strip=True) if prev_header else ""
            level_records = parse_detail_table(tbl)
            if level_records:
                piece_df = pd.DataFrame(level_records)
                piece_df.attrs["section_title"] = section
                pieces[piece_name] = piece_df
        return pieces

    # --- 1. Scraping armature ---
    print("Scaricamento pagina armature...")
    response = requests.get(f"{base_url}/wikis/god-of-war-ragnarok/All_Armor_Sets", headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    all_tables = soup.find_all("table")

    dfs = []
    for idx, tbl in enumerate(all_tables):
        try:
            t = pd.read_html(StringIO(str(tbl)))[0]
            dfs.append(t)
        except Exception:
            pass
    df = max(dfs, key=lambda x: x.shape[0])

    target_table = all_tables[1]
    rows = target_table.find_all("tr")[1:]
    armor_data = []
    for row in rows:
        cells = row.find_all(["td", "th"])
        if cells:
            link_tag = cells[0].find("a")
            name = link_tag.get_text(strip=True) if link_tag else cells[0].get_text(strip=True)
            href = None
            if link_tag and link_tag.get("href"):
                href = link_tag["href"]
                if not href.startswith("http"):
                    href = base_url + href
            armor_data.append({"Set Name": name, "URL": href})
    df_links = pd.DataFrame(armor_data)
    df["Set Name"] = df_links["Set Name"]
    df["URL"] = df_links["URL"]
    print(f"  Set trovati: {len(df)}, con link: {df['URL'].notna().sum()}")

    armor_details = {}
    errors = []
    valid_sets = df[df["URL"].notna()]
    total = len(valid_sets)
    for i, (_, row) in enumerate(valid_sets.iterrows()):
        name = row["Set Name"]
        url = row["URL"]
        print(f"  [{i+1}/{total}] {name}...", end=" ")
        try:
            pieces = scrape_armor_detail(url)
            armor_details[name] = pieces
            print(f"OK ({len(pieces)} pezzi)")
        except Exception as e:
            errors.append((name, str(e)))
            print(f"ERRORE: {e}")
        time.sleep(0.5)
    print(f"  Completato: {len(armor_details)}/{total} OK, {len(errors)} errori\n")

    records = []
    for set_name, pieces in armor_details.items():
        for piece_type, piece_df in pieces.items():
            section_title = piece_df.attrs.get("section_title", "")
            for _, row_data in piece_df.iterrows():
                record = {"Set Name": set_name, "Piece Type": piece_type, "Piece Name": section_title}
                for col in piece_df.columns:
                    record[col] = row_data[col]
                records.append(record)
    all_pieces_df = pd.DataFrame(records)
    all_pieces_df["Level"] = pd.to_numeric(all_pieces_df["Level"], errors="coerce")
    all_pieces_df["Total Stats"] = all_pieces_df[stat_cols].sum(axis=1)
    print(f"  all_pieces_df: {all_pieces_df.shape[0]} righe, {all_pieces_df['Piece Name'].nunique()} pezzi")

    # --- 2. Scraping armi ---
    print("\nScaricamento pagina armi...")
    resp_weapons = requests.get(f"{base_url}/wikis/god-of-war-ragnarok/All_Weapon_and_Shield_Attachments",
                                headers=headers)
    resp_weapons.raise_for_status()
    soup_weapons = BeautifulSoup(resp_weapons.text, "lxml")
    weapon_tables = soup_weapons.find_all("table")

    weapon_categories = {1: "Leviathan Axe", 2: "Blades of Chaos", 3: "Draupnir Spear", 4: "Shield"}
    weapon_data = []
    for tbl_idx, category in weapon_categories.items():
        tbl_tag = weapon_tables[tbl_idx]
        tbl_rows = tbl_tag.find_all("tr")[1:]
        for row in tbl_rows:
            cells = row.find_all(["td", "th"])
            if cells:
                link_tag = cells[0].find("a")
                name = link_tag.get_text(strip=True) if link_tag else cells[0].get_text(strip=True)
                href = None
                if link_tag and link_tag.get("href"):
                    href = link_tag["href"]
                    if not href.startswith("http"):
                        href = base_url + href
                weapon_data.append({"Category": category, "Name": name, "URL": href})
    weapons_df = pd.DataFrame(weapon_data)
    print(f"  Attachment trovati: {len(weapons_df)}")

    weapon_details = {}
    weapon_errors = []
    total_w = len(weapons_df)
    for i, (_, row) in enumerate(weapons_df.iterrows()):
        name = row["Name"]
        cat = row["Category"]
        url = row["URL"]
        print(f"  [{i+1}/{total_w}] {cat} — {name}...", end=" ")
        try:
            resp_w = requests.get(url, headers=headers)
            resp_w.raise_for_status()
            soup_w = BeautifulSoup(resp_w.text, "lxml")
            tbls = soup_w.find_all("table")
            if tbls:
                records_w = parse_detail_table(tbls[0])
                if records_w:
                    wdf = pd.DataFrame(records_w)
                    wdf["Level"] = pd.to_numeric(wdf["Level"], errors="coerce")
                    weapon_details[(cat, name)] = wdf
                    print(f"OK ({len(records_w)} livelli)")
                else:
                    print("VUOTO")
            else:
                print("VUOTO")
        except Exception as e:
            weapon_errors.append((name, str(e)))
            print(f"ERRORE: {e}")
        time.sleep(0.5)
    print(f"  Completato: {len(weapon_details)}/{total_w} OK, {len(weapon_errors)} errori\n")

    weapon_records = []
    for (category, weapon_name), wdf in weapon_details.items():
        for _, row_data in wdf.iterrows():
            record = {"Category": category, "Weapon Name": weapon_name}
            for col in wdf.columns:
                record[col] = row_data[col]
            weapon_records.append(record)
    all_weapons_df = pd.DataFrame(weapon_records)
    all_weapons_df["Level"] = pd.to_numeric(all_weapons_df["Level"], errors="coerce")
    for c in stat_cols:
        if c not in all_weapons_df.columns:
            all_weapons_df[c] = 0
    all_weapons_df[stat_cols] = all_weapons_df[stat_cols].fillna(0)
    # Fix: Soldier's Sauroter Level 9.1
    fix_mask = (all_weapons_df["Weapon Name"] == "Soldier's Sauroter") & (all_weapons_df["Level"] == 9.1)
    all_weapons_df.loc[fix_mask, "Cooldown"] = 19
    all_weapons_df.loc[fix_mask, "Luck"] = 19
    all_weapons_df["Total Stats"] = all_weapons_df[stat_cols].sum(axis=1)
    print(f"  all_weapons_df: {all_weapons_df.shape[0]} righe, {all_weapons_df['Weapon Name'].nunique()} armi")

    # --- 3. Salvataggio CSV ---
    all_pieces_df.to_csv(ARMOR_CSV, index=False)
    all_weapons_df.to_csv(WEAPONS_CSV, index=False)
    print(f"\nCSV salvati: {ARMOR_CSV}, {WEAPONS_CSV}")

Caricati da CSV:
  Armature: 756 righe, 81 pezzi unici
  Armi:     439 righe, 46 armi uniche


In [33]:
# Costruzione inventario armature da configurazione
# Formato: (nome, livello, tipo_pezzo, needs_craft)
def _parse_inv(pieces, slot_type):
    return [(t[0], t[1], slot_type, len(t) > 2 and t[2] == "craft") for t in pieces]

inventory = (
    _parse_inv(chest_pieces, "Chest")
    + _parse_inv(wrist_pieces, "Wrist")
    + _parse_inv(waist_pieces, "Waist")
)

filters = []
for piece_name, max_lvl, piece_type, needs_craft in inventory:
    mask = (
        (all_pieces_df["Piece Name"] == piece_name)
        & (all_pieces_df["Piece Type"] == piece_type)
        & (all_pieces_df["Level"] <= max_lvl)
    )
    filters.append(mask)

available_df = all_pieces_df[pd.concat(filters, axis=1).any(axis=1)].copy()
craft_count = sum(1 for *_, c in inventory if c)
print(f"Armature in inventario: {len(inventory)} pezzi ({craft_count} da craftare), {available_df.shape[0]} righe DB")

Armature in inventario: 29 pezzi (18 da craftare), 37 righe DB


In [34]:
# Costruzione inventario armi da configurazione
# Formato: (nome, livello, categoria, needs_craft)
w_inventory = (
    _parse_inv(axe_attachments, "Leviathan Axe")
    + _parse_inv(blades_attachments, "Blades of Chaos")
    + _parse_inv(spear_attachments, "Draupnir Spear")
    + _parse_inv(shield_attachments, "Shield")
)

if w_inventory:
    w_filters = []
    for wname, max_lvl, cat, needs_craft in w_inventory:
        mask = (
            (all_weapons_df["Weapon Name"] == wname)
            & (all_weapons_df["Category"] == cat)
            & (all_weapons_df["Level"] <= max_lvl)
        )
        w_filters.append(mask)
    w_available_df = all_weapons_df[pd.concat(w_filters, axis=1).any(axis=1)].copy()
    craft_count_w = sum(1 for *_, c in w_inventory if c)
    print(f"Armi in inventario: {len(w_inventory)} attachment ({craft_count_w} da craftare), {w_available_df.shape[0]} righe DB")
else:
    w_available_df = pd.DataFrame()
    print("Inventario armi vuoto.")

Armi in inventario: 14 attachment (3 da craftare), 23 righe DB


## Risultati

Build attuale e piano di upgrade ottimo con vincoli di risorse.

In [35]:
stat_cols = ["Strength", "Defense", "Runic", "Vitality", "Cooldown", "Luck"]

# ── Raccolta pezzi armatura al livello attuale (solo posseduti, no craft) ──
inv_lookup = {(name, pt): (lvl, craft) for name, lvl, pt, craft in inventory}
armor_current = []
for (piece_name, piece_type), (max_lvl, needs_craft) in inv_lookup.items():
    if needs_craft:
        continue  # non posseduto, escluso dal build attuale
    row = available_df[
        (available_df["Piece Name"] == piece_name)
        & (available_df["Piece Type"] == piece_type)
        & (available_df["Level"] == max_lvl)
    ]
    if not row.empty:
        r = row.iloc[0].copy()
        r["Slot"] = f"Armatura — {piece_type}"
        r["Item Name"] = piece_name
        r["Item Level"] = max_lvl
        armor_current.append(r)

# ── Raccolta armi al livello attuale (solo possedute, no craft) ──
w_inv_lookup = {(name, cat): (lvl, craft) for name, lvl, cat, craft in w_inventory}
weapon_current = []
for (wname, cat), (max_lvl, needs_craft) in w_inv_lookup.items():
    if needs_craft:
        continue  # non posseduto, escluso dal build attuale
    row = w_available_df[
        (w_available_df["Weapon Name"] == wname)
        & (w_available_df["Category"] == cat)
        & (w_available_df["Level"] == max_lvl)
    ]
    if not row.empty:
        r = row.iloc[0].copy()
        r["Slot"] = f"Arma — {cat}"
        r["Item Name"] = wname
        r["Item Level"] = max_lvl
        weapon_current.append(r)

# ═══════════════════════════════════════════════════════════════
# 1. BUILD OTTIMALE COMBINATA
# ═══════════════════════════════════════════════════════════════
print("=" * 85)
print("  BUILD OTTIMALE COMPLETA (Armatura + Armi)")
print("=" * 85)

armor_slots = {"Chest": "Armatura — Chest", "Wrist": "Armatura — Wrist", "Waist": "Armatura — Waist"}
weapon_slots = {
    "Leviathan Axe": "Arma — Leviathan Axe",
    "Blades of Chaos": "Arma — Blades of Chaos",
    "Draupnir Spear": "Arma — Draupnir Spear",
    "Shield": "Arma — Shield",
}

build_items = []

print(f"\n  {'─'*40}")
print(f"  ARMATURA")
print(f"  {'─'*40}")
armor_total = 0
for pt in ["Chest", "Wrist", "Waist"]:
    candidates = [r for r in armor_current if r["Piece Type"] == pt]
    if candidates:
        best = max(candidates, key=lambda x: x["Total Stats"])
        stats_detail = ", ".join(f"{s[:3].upper()}:{best.get(s,0):.0f}" for s in stat_cols if best.get(s,0) > 0)
        print(f"    {pt:6s}: {best['Item Name']} (LVL {best['Item Level']:.0f}) — "
              f"Total: {best['Total Stats']:.0f}  [{stats_detail}]")
        armor_total += best["Total Stats"]
        build_items.append({"Slot": armor_slots[pt], "Item": best["Item Name"],
                            "Level": best["Item Level"], "Total Stats": best["Total Stats"]})

print(f"\n  {'─'*40}")
print(f"  ARMI")
print(f"  {'─'*40}")
weapon_total = 0
for cat in ["Leviathan Axe", "Blades of Chaos", "Draupnir Spear", "Shield"]:
    candidates = [r for r in weapon_current if r["Category"] == cat]
    if candidates:
        best = max(candidates, key=lambda x: x["Total Stats"])
        stats_detail = ", ".join(f"{s[:3].upper()}:{best.get(s,0):.0f}" for s in stat_cols if best.get(s,0) > 0)
        print(f"    {cat:18s}: {best['Item Name']} (LVL {best['Item Level']:.0f}) — "
              f"Total: {best['Total Stats']:.0f}  [{stats_detail}]")
        weapon_total += best["Total Stats"]
        build_items.append({"Slot": weapon_slots[cat], "Item": best["Item Name"],
                            "Level": best["Item Level"], "Total Stats": best["Total Stats"]})
    else:
        print(f"    {cat:18s}: — non sbloccata —")

grand_total = armor_total + weapon_total

print(f"\n  {'═'*40}")
print(f"  TOTALE ARMATURA:  {armor_total:.0f}")
print(f"  TOTALE ARMI:      {weapon_total:.0f}")
print(f"  ────────────────────────────────────────")
print(f"  GRAND TOTAL:      {grand_total:.0f}")
print(f"  {'═'*40}")

# Segnala pezzi da craftare
craft_a = [(n, l, pt) for n, l, pt, c in inventory if c]
craft_w = [(n, l, cat) for n, l, cat, c in w_inventory if c]
if craft_a or craft_w:
    print(f"\n  ⚠ Pezzi da craftare (non inclusi nel build attuale):")
    for n, l, pt in craft_a:
        print(f"    {n} ({pt}) — LVL {l}")
    for n, l, cat in craft_w:
        print(f"    {n} ({cat}) — LVL {l}")

# ═══════════════════════════════════════════════════════════════
# 2. CLASSIFICA COMPLETA PER SLOT
# ═══════════════════════════════════════════════════════════════
print(f"\n\n{'='*85}")
print("  CLASSIFICA COMPLETA PER SLOT")
print(f"{'='*85}")

for pt in ["Chest", "Wrist", "Waist"]:
    candidates = sorted([r for r in armor_current if r["Piece Type"] == pt],
                         key=lambda x: x["Total Stats"], reverse=True)
    if candidates:
        print(f"\n  ARMATURA — {pt.upper()}:")
        for i, r in enumerate(candidates):
            stats_detail = ", ".join(f"{s[:3].upper()}:{r.get(s,0):.0f}" for s in stat_cols if r.get(s,0) > 0)
            marker = "→" if i == 0 else " "
            star = "  ★" if i == 0 else ""
            print(f"    {marker} {r['Item Name']} (LVL {r['Item Level']:.0f}) — "
                  f"Total: {r['Total Stats']:.0f}  [{stats_detail}]{star}")

for cat in ["Leviathan Axe", "Blades of Chaos", "Draupnir Spear", "Shield"]:
    candidates = sorted([r for r in weapon_current if r["Category"] == cat],
                         key=lambda x: x["Total Stats"], reverse=True)
    if candidates:
        print(f"\n  ARMA — {cat.upper()}:")
        for i, r in enumerate(candidates):
            stats_detail = ", ".join(f"{s[:3].upper()}:{r.get(s,0):.0f}" for s in stat_cols if r.get(s,0) > 0)
            marker = "→" if i == 0 else " "
            star = "  ★" if i == 0 else ""
            print(f"    {marker} {r['Item Name']} (LVL {r['Item Level']:.0f}) — "
                  f"Total: {r['Total Stats']:.0f}  [{stats_detail}]{star}")

  BUILD OTTIMALE COMPLETA (Armatura + Armi)

  ────────────────────────────────────────
  ARMATURA
  ────────────────────────────────────────
    Chest : Spiritual Shoulder Straps (LVL 4) — Total: 78  [STR:28, DEF:22, COO:28]
    Wrist : Spiritual Wraps (LVL 5) — Total: 64  [STR:41, COO:23]
    Waist : Spiritual Belt (LVL 5) — Total: 64  [DEF:41, COO:23]

  ────────────────────────────────────────
  ARMI
  ────────────────────────────────────────
    Leviathan Axe     : Grip of the Fallen Alchemist (LVL 5) — Total: 39  [STR:15, VIT:12, LUC:12]
    Blades of Chaos   : Hardened War Handles (LVL 5) — Total: 38  [STR:20, DEF:9, VIT:9]
    Draupnir Spear    : — non sbloccata —
    Shield            : Rond of Affliction (LVL 5) — Total: 39  [DEF:15, RUN:12, COO:12]

  ════════════════════════════════════════
  TOTALE ARMATURA:  206
  TOTALE ARMI:      116
  ────────────────────────────────────────
  GRAND TOTAL:      322
  ════════════════════════════════════════

  ⚠ Pezzi da craftare (non 

In [36]:
from collections import Counter

stat_cols = ["Strength", "Defense", "Runic", "Vitality", "Cooldown", "Luck"]

# ─── Normalizzazione nomi materiali (typo/plurali nel DB) ───
MAT_ALIASES = {
    "Smouldering Embers": "Smoldering Embers",
    "Petrified Bones": "Petrified Bone",
    "Whispering Slabs": "Whispering Slab",
    "Asgardian Ingots": "Asgardian Ingot",
    "Dwaren Steel": "Dwarven Steel",
    "s Broken Cuirass": "Lunda's Broken Cuirass",
    "s Broken Bracers": "Lunda's Broken Bracers",
    "s Broken Belt": "Lunda's Broken Belt",
}

def normalize_mat(name):
    return MAT_ALIASES.get(name, name)

def get_available(mat_name):
    return resource_budget.get(normalize_mat(mat_name), 0)


# ─── Funzioni helper con vincoli materiali ───

def get_upgrade_chain_with_mats(df, name_col, name, cat_col, cat, current_lvl):
    """Per un item, restituisce [(target_level, total_stats, cum_hack, cum_mats)]
    Tronca la catena al primo livello che richiede materiali non disponibili.
    Per item da craftare, passare current_lvl < base_level per includere il costo di crafting."""
    upg_cols = [c for c in df.columns if c.startswith("Upgrade_") and c != "Upgrade_Hacksilver"]
    item_df = df[(df[name_col] == name) & (df[cat_col] == cat)].sort_values("Level")
    chain = []
    cum_hack = 0
    cum_mats = Counter()

    for _, r in item_df.iterrows():
        if r["Level"] <= current_lvl:
            continue
        hack = int(r.get("Upgrade_Hacksilver", 0)) if pd.notna(r.get("Upgrade_Hacksilver", 0)) else 0
        level_mats = {}
        for c in upg_cols:
            v = r.get(c, 0)
            if pd.notna(v) and v > 0:
                mat_name = normalize_mat(c.replace("Upgrade_", ""))
                level_mats[mat_name] = level_mats.get(mat_name, 0) + int(v)

        test_mats = Counter(cum_mats)
        test_mats.update(level_mats)
        feasible = all(test_mats[m] <= get_available(m) for m in test_mats)
        if not feasible:
            break

        cum_hack += hack
        cum_mats = test_mats
        chain.append((r["Level"], r["Total Stats"], cum_hack, dict(cum_mats)))

    return chain


def build_slot_options_with_mats(items_with_chains):
    """Per uno slot, genera (hack, stats, label, mats_dict) per ogni azione possibile."""
    current_best = max((s for _, _, s, _, _ in items_with_chains), default=0)
    options = [(0, current_best, "— nessuna azione —", {})]

    for item_name, item_lvl, item_stats, chain, needs_craft in items_with_chains:
        other_best = max((s for n, _, s, _, _ in items_with_chains if n != item_name), default=0)
        for target_lvl, target_stats, hack, mats in chain:
            lvl_label = int(target_lvl) if target_lvl == int(target_lvl) else target_lvl
            resulting_stats = max(target_stats, other_best)
            craft_tag = "★craft+" if needs_craft else ""
            options.append((hack, resulting_stats,
                            f"{craft_tag}{item_name} {int(item_lvl)}→{lvl_label}", mats))
    return options


def pareto_frontier_with_mats(options):
    """Opzioni non-dominate ordinate per costo hacksilver crescente."""
    frontier, best_stats = [], -1
    for opt in sorted(options, key=lambda x: (x[0], -x[1])):
        if opt[1] > best_stats:
            frontier.append(opt)
            best_stats = opt[1]
    return frontier


def solve_with_resources(slot_pareto_dict, budget_hack):
    """Enumerazione con pruning: per ogni combinazione di opzioni (una per slot),
    verifica che il totale Hacksilver + tutti i materiali rientrino nel budget."""
    slots = list(slot_pareto_dict.keys())
    best_total, best_choices = -1, {}

    slot_opts = []
    for slot in slots:
        feasible = [(h, s, l, m) for h, s, l, m in slot_pareto_dict[slot] if h <= budget_hack]
        slot_opts.append(feasible)

    order = sorted(range(len(slots)), key=lambda i: len(slot_opts[i]))

    def search(idx, used_hack, used_mats, acc_stats, acc_choices):
        nonlocal best_total, best_choices
        if idx == len(slots):
            if acc_stats > best_total:
                best_total = acc_stats
                best_choices = dict(acc_choices)
            return

        si = order[idx]
        slot = slots[si]
        for hack, stats, label, mats in slot_opts[si]:
            new_hack = used_hack + hack
            if new_hack > budget_hack:
                continue
            new_mats = Counter(used_mats)
            new_mats.update(mats)
            if all(new_mats[m] <= get_available(m) for m in new_mats):
                acc_choices[slot] = (hack, stats, label, mats)
                search(idx + 1, new_hack, new_mats, acc_stats + stats, acc_choices)
                del acc_choices[slot]

    search(0, 0, Counter(), 0, {})
    return best_total, best_choices


# ─── Costruzione opzioni per slot ───

slot_pareto = {}

for pt in ["Chest", "Wrist", "Waist"]:
    items = []
    for name, lvl, t, needs_craft in inventory:
        if t != pt:
            continue
        if needs_craft:
            # Item da craftare: stats attuali = 0, chain include il costo di crafting
            effective_lvl = lvl - 1  # include il livello base (craft cost)
            stats = 0
        else:
            effective_lvl = lvl
            row = all_pieces_df[
                (all_pieces_df["Piece Name"] == name) & (all_pieces_df["Piece Type"] == pt)
                & (all_pieces_df["Level"] == lvl)
            ]
            stats = row.iloc[0]["Total Stats"] if not row.empty else 0
        chain = get_upgrade_chain_with_mats(all_pieces_df, "Piece Name", name, "Piece Type", pt, effective_lvl)
        items.append((name, lvl, stats, chain, needs_craft))
    slot_pareto[f"Armatura — {pt}"] = pareto_frontier_with_mats(build_slot_options_with_mats(items))

for cat in ["Leviathan Axe", "Blades of Chaos", "Shield"]:
    items = []
    for name, lvl, c, needs_craft in w_inventory:
        if c != cat:
            continue
        if needs_craft:
            effective_lvl = lvl - 1
            stats = 0
        else:
            effective_lvl = lvl
            row = all_weapons_df[
                (all_weapons_df["Weapon Name"] == name) & (all_weapons_df["Category"] == cat)
                & (all_weapons_df["Level"] == lvl)
            ]
            stats = row.iloc[0]["Total Stats"] if not row.empty else 0
        chain = get_upgrade_chain_with_mats(all_weapons_df, "Weapon Name", name, "Category", cat, effective_lvl)
        items.append((name, lvl, stats, chain, needs_craft))
    if items:
        slot_pareto[f"Arma — {cat}"] = pareto_frontier_with_mats(build_slot_options_with_mats(items))


# ─── Report ───

print("=" * 85)
print("  OTTIMIZZATORE DI BUILD (con vincoli materiali)")
print("=" * 85)
print(f"  Grand Total attuale: {grand_total:.0f}")
print(f"  Hacksilver disponibili: {resource_budget['Hacksilver']:,}\n")

print("  RISORSE DISPONIBILI:")
for mat, qty in sorted(resource_budget.items()):
    if mat != "Hacksilver":
        print(f"    {mat}: {qty}")
print()

print("  FRONTIERE DI PARETO PER SLOT:\n")
for slot, frontier in slot_pareto.items():
    real_options = [f for f in frontier if "nessuna" not in f[2]]
    print(f"  {slot}: {len(real_options)} upgrade possibili")
    for hack, stats, label, mats in frontier:
        if "nessuna" in label:
            continue
        mat_str = ", ".join(f"{m}:{v}" for m, v in sorted(mats.items())) if mats else "—"
        print(f"    Hack {hack:>8,} → Stats {stats:>6.0f}  [{label}]  Mat: {mat_str}")
    if not real_options:
        print(f"    (nessun upgrade fattibile con i materiali attuali)")
    print()

# ─── Soluzione ottima ───

print(f"\n{'='*85}")
print("  PIANO OTTIMO (Hacksilver + Materiali)")
print(f"{'='*85}\n")

total_stats, choices = solve_with_resources(slot_pareto, resource_budget["Hacksilver"])
total_hack = sum(h for h, _, _, _ in choices.values())
total_mats = Counter()
for h, s, l, m in choices.values():
    total_mats.update(m)

gain = total_stats - grand_total
actions = [(sl, h, s, l, m) for sl, (h, s, l, m) in sorted(choices.items()) if "nessuna" not in l]

if gain > 0:
    print(f"  Grand Total: {total_stats:.0f} (+{gain:.0f})")
    print(f"  Hacksilver spesi: {total_hack:,} / {resource_budget['Hacksilver']:,} "
          f"(restano {resource_budget['Hacksilver'] - total_hack:,})")
    if total_mats:
        print(f"  Materiali usati:")
        for m, v in sorted(total_mats.items()):
            avail = get_available(m)
            print(f"    {m}: {v}/{avail}")
    print()
    for sl, h, s, l, m in actions:
        mat_str = ", ".join(f"{k}:{v}" for k, v in sorted(m.items())) if m else "solo Hacksilver"
        print(f"  ► {l:40s}  Hack: {h:>8,}  → Slot: {s:.0f}  [{mat_str}]")
else:
    print("  Nessun upgrade possibile con le risorse attuali.")

# ─── Slot bloccati ───

print(f"\n\n{'='*85}")
print("  SLOT BLOCCATI — materiali mancanti")
print(f"{'='*85}\n")

for pt in ["Chest", "Wrist", "Waist"]:
    for name, lvl, t, needs_craft in inventory:
        if t != pt: continue
        if needs_craft:
            # Per item da craftare, il "prossimo livello" da verificare è il livello base (craft cost)
            next_lvl = lvl
        else:
            next_lvl = lvl + 1
        row = all_pieces_df[(all_pieces_df["Piece Name"]==name)&(all_pieces_df["Piece Type"]==pt)&(all_pieces_df["Level"]==next_lvl)]
        if not row.empty:
            upg_cols = [c for c in all_pieces_df.columns if c.startswith("Upgrade_") and c!="Upgrade_Hacksilver"]
            blocking = []
            for c in upg_cols:
                v = row.iloc[0].get(c,0)
                if pd.notna(v) and v > 0:
                    mn = normalize_mat(c.replace("Upgrade_",""))
                    avail = get_available(mn)
                    if avail < v:
                        tag = " (craft)" if needs_craft else ""
                        blocking.append(f"{mn} ({int(v)} richiesti, {avail} disponibili)")
            if blocking:
                action = f"craft LVL {lvl}" if needs_craft else f"{lvl}→{next_lvl}"
                print(f"  {name} {action}: manca {', '.join(blocking)}")

for cat_w in ["Leviathan Axe", "Blades of Chaos", "Shield"]:
    for name, lvl, c, needs_craft in w_inventory:
        if c != cat_w: continue
        if needs_craft:
            next_lvl = lvl
        else:
            next_lvl = lvl + 1
        row = all_weapons_df[(all_weapons_df["Weapon Name"]==name)&(all_weapons_df["Category"]==cat_w)&(all_weapons_df["Level"]==next_lvl)]
        if not row.empty:
            upg_cols = [c2 for c2 in all_weapons_df.columns if c2.startswith("Upgrade_") and c2!="Upgrade_Hacksilver"]
            blocking = []
            for c2 in upg_cols:
                v = row.iloc[0].get(c2,0)
                if pd.notna(v) and v > 0:
                    mn = normalize_mat(c2.replace("Upgrade_",""))
                    avail = get_available(mn)
                    if avail < v:
                        blocking.append(f"{mn} ({int(v)} richiesti, {avail} disponibili)")
            if blocking:
                action = f"craft LVL {lvl}" if needs_craft else f"{lvl}→{next_lvl}"
                print(f"  {name} {action}: manca {', '.join(blocking)}")

# ─── Sequenza step-by-step ───

print(f"\n\n{'='*85}")
print("  SEQUENZA STEP-BY-STEP (ordine per efficienza, con vincoli materiali)")
print(f"{'='*85}\n")

remaining_slots = {slot: list(opts) for slot, opts in slot_pareto.items()}
cur_stats = {slot: opts[0][1] for slot, opts in slot_pareto.items()}
running_total = grand_total
used_budget = 0
used_mats_total = Counter()

for step_i in range(1, 50):
    best_action = None
    best_efficiency = -1
    for slot, opts in remaining_slots.items():
        cs = cur_stats[slot]
        for hack, stats, label, mats in opts:
            if hack == 0 or stats <= cs:
                continue
            test = Counter(used_mats_total)
            test.update(mats)
            if not all(test[m] <= get_available(m) for m in test):
                continue
            if used_budget + hack > resource_budget["Hacksilver"]:
                continue
            g = stats - cs
            eff = g / hack * 1000
            if eff > best_efficiency:
                best_efficiency = eff
                best_action = (slot, hack, stats, label, g, eff, mats)

    if best_action is None:
        break

    slot, hack, stats, label, gain, eff, mats = best_action
    used_budget += hack
    running_total += gain
    cur_stats[slot] = stats
    used_mats_total.update(mats)
    remaining_slots[slot] = [(h, s, l, m) for h, s, l, m in remaining_slots[slot] if s > stats]

    mat_str = ", ".join(f"{k}:{v}" for k, v in sorted(mats.items())) if mats else "—"
    print(f"  {step_i:2d}. {label}")
    print(f"      [{slot}]  +{gain:.0f} stats  │  Hack: {hack:>8,}  │  "
          f"Eff: {eff:.2f}/1k  │  GT: {running_total:.0f}  │  Cum: {used_budget:,}")
    print(f"      Materiali: {mat_str}")
    print()

print(f"  {'═'*70}")
print(f"  Grand Total finale: {running_total:.0f}  (+{running_total - grand_total:.0f})")
print(f"  Hacksilver spesi:   {used_budget:,} / {resource_budget['Hacksilver']:,}")
print(f"  Hacksilver restanti: {resource_budget['Hacksilver'] - used_budget:,}")
if used_mats_total:
    print(f"  Materiali consumati:")
    for m, v in sorted(used_mats_total.items()):
        avail = get_available(m)
        print(f"    {m}: {v}/{avail} (restano {avail - v})")
print(f"  {'═'*70}")

  OTTIMIZZATORE DI BUILD (con vincoli materiali)
  Grand Total attuale: 322
  Hacksilver disponibili: 2,183

  RISORSE DISPONIBILI:
    Asgardian Ingot: 1
    Blazing Embers: 0
    Bonded Leather: 35
    Celestial Fossil: 0
    Divine Ashes: 0
    Dragon Claw: 0
    Dragon Tooth: 4
    Dust of Realms: 0
    Dwarven Steel: 12
    Essence of Hel: 0
    Forged Iron: 60
    Forsaken Breath: 0
    Fortified Remnants: 0
    Gleaming Crystal: 0
    Glowing Embers: 0
    Hardened Remnants: 0
    Honed Metal: 10
    Lindwyrm Scales: 0
    Luminous Alloy: 0
    Lunda's Broken Belt: 0
    Lunda's Broken Bracers: 0
    Lunda's Broken Cuirass: 0
    Mountain Root: 0
    Nidavellir Ore: 2
    Petrified Bone: 0
    Purified Crystalline: 0
    Rawhide: 36
    Shining Crystal: 0
    Skap Slag: 0
    Slag Deposits: 25
    Smoldering Embers: 0
    Sovereign Coals: 0
    Sparkling Crystal: 0
    Stonewood: 12
    Tempered Remnants: 0
    Whispering Slab: 52

  FRONTIERE DI PARETO PER SLOT:

  Armatura — C